In [0]:
%run /Users/nethumgimsara605@gmail.com/ecommerce-lakehouse-databricks-repo/01-ingestion/setup-storage-connection

In [0]:
from pyspark.sql.functions import count, row_number, col
from pyspark.sql.window import Window

In [0]:
bronze_products = spark.read.format("parquet").load("abfss://bronze@ecommercelakehouse01.dfs.core.windows.net/products/")
bronze_products.show(5)
print(f"Bronze products count: {bronze_products.count()}")

In [0]:
duplicate_check = bronze_products.groupby("product_id").agg(count("*").alias("cnt")).filter("cnt > 1")
duplicate_count = duplicate_check.count()
print(f"Duplicate products_ids found: {duplicate_count}")

In [0]:
window_spec = Window.partitionBy("product_id").orderBy("product_name")
deduped_products = bronze_products.withColumn("row_num", row_number().over(window_spec)) \
    .filter("row_num = 1") \
    .drop("row_num")

print(f"Rows before dedupe: {bronze_products.count()}, after: {deduped_products.count()}")

In [0]:
quarantined_nulls = deduped_products.filter(
    "product_id IS NULL OR product_name IS NULL OR category IS NULL OR price IS NULL"
)
quarantined_nulls.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/_quarantine/products_null_violations/")

cleaned_products = deduped_products.filter(
    "product_id IS NOT NULL OR product_name IS NOT NULL OR category IS NOT NULL OR price IS NOT NULL"
)

print(f"Qurantined for nulls: {quarantined_nulls.count()}, REMANING: {cleaned_products.count()}")

In [0]:
quarantined_bad_price = cleaned_products.filter("price <= 0")
quarantined_bad_price.write.format("delta").mode("append").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/_quarantine/products_price_violations/")

cleaned_products = cleaned_products.filter("price > 0")

print(f"Qurantined for bad price: {quarantined_bad_price.count()}, final clean count: {cleaned_products.count()}")

In [0]:
cleaned_products.write.format("delta").mode("overwrite").save("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/products/")

In [0]:
silver_products_check = spark.read.format("delta").load("abfss://silver@ecommercelakehouse01.dfs.core.windows.net/products/")
silver_products_check.printSchema()
print(f"Total rows in silver products: {silver_products_check.count()}")
silver_products_check.show(5)